[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sear-labs/energy-system-modeling/blob/main/notebooks/graduate/model_diversity.ipynb)

# GRAD-M: Model Diversity — the Same System in a Second Tool
## REE 4301 / IE 5300 / IE 6301 — Energy Systems Modeling
### Graduate sections only. Individual work. After SB6.

Every model in this course has been PyPSA. That is a problem, because you have no way to tell which of your results are properties of *the system* and which are properties of *the tool*.

This assignment separates them. You will take a system you have already solved, express its inputs in a tool-neutral form, rebuild the identical optimisation problem in a second solver stack, and then account for every dollar of difference between the two answers.

**The claim you have to be able to defend at the end:** *this much of the difference is a data assumption, this much is model structure, and this much was a bookkeeping convention I had not noticed.*

### Which second tool

The assignment names **PowerGenome → GenX**. That is the full-weight route, it is what a research group would actually do, and it is documented in Part 6 — but GenX is Julia and PowerGenome is a substantial data pipeline, so neither runs inside this notebook.

The route taken here is the sanctioned lighter one: **OSeMOSYS via PuLP** — which, stripped of its own conventions, means *writing the capacity-expansion linear program yourself, from the algebra, in a different modelling library*. That is more work than downloading a second tool and it teaches more, because a difference you cannot explain is a difference in something you wrote.

**You may substitute GenX, OSeMOSYS-Pyomo, or the real OSeMOSYS_PuLP distribution if you prefer.** The deliverable is the decomposition, not the tool.


In [1]:
!pip install -q pypsa highspy pulp gurobipy

In [2]:
import numpy as np
import pandas as pd
import pypsa
import pulp

HOURS = 24
HUBS = ['West Texas', 'Dallas-Fort Worth', 'Houston', 'Austin',
        'San Antonio']
SHARE = {'West Texas': 0.127, 'Dallas-Fort Worth': 0.341,
         'Houston': 0.299, 'Austin': 0.065, 'San Antonio': 0.168}
PEAK_MW = 45_000.0

# (from, to, km) — great-circle distances between the real hub coordinates
ARCS = [('West Texas', 'Dallas-Fort Worth', 503),
        ('West Texas', 'Austin', 455),
        ('Dallas-Fort Worth', 'Houston', 362),
        ('Dallas-Fort Worth', 'Austin', 293),
        ('Houston', 'Austin', 235),
        ('Austin', 'San Antonio', 118),
        ('Houston', 'San Antonio', 304)]
WIND_MAX = {'West Texas': 22_000, 'San Antonio': 4_000,
            'Dallas-Fort Worth': 3_000}
SOLAR_MAX = {'West Texas': 18_000, 'San Antonio': 4_000,
             'Dallas-Fort Worth': 4_000, 'Austin': 2_000, 'Houston': 3_000}
GAS_MAX = {'Houston': 30_000, 'Dallas-Fort Worth': 12_000}
GAS_MC = 22.14        # $/MWh-e delivered — from the SB6 unit check
LINE_CAPEX_PER_MW_KM, LINE_LIFE = 1200.0, 40
EXISTING_ARC_MW = 3000.0


def crf(rate, years):
    return rate * (1 + rate) ** years / ((1 + rate) ** years - 1)


def daily_capital(capex, life, wacc):
    return capex * crf(wacc, life) / 365.0


def profiles(seed=42):
    h = np.arange(HOURS)
    shape = np.clip(0.60 + 0.40 * np.sin(np.pi * (h - 6) / 12)
                    * ((h >= 6) & (h <= 22)), 0.4, 1.0)
    shape = shape / shape.max()
    solar = np.zeros(HOURS)
    for k in range(6, 19):
        solar[k] = np.sin(np.pi * (k - 6) / 13)
    rng = np.random.default_rng(seed)
    wind = np.clip(0.35 + 0.25 * np.cos(2 * np.pi * h / 24)
                   + 0.10 * rng.normal(0, 1, HOURS), 0.05, 0.85)
    return shape, np.clip(solar, 0, 1), wind


---

### Choosing a solver



This notebook defaults to **Gurobi**. `pip install gurobipy` ships a restricted licence that needs no registration at all and solves models up to **2,000 variables and 2,000 constraints**.



That ceiling arrives sooner than you would think. Measured sizes for the models in this course:



| model | variables | constraints | restricted licence |

|---|---|---|---|

| 1-node, 24 hours | 123 | 291 | fits |

| SB6 Stage 1 | 766 | 1,837 | fits |

| SB6 Stage 2 | 886 | 2,172 | too big |

| SB6 Stage 3 | 2,614 | 6,444 | too big |

| TX-123BT, 24 hours | 16,080 | 38,304 | 19x over |



When you exceed it, Gurobi returns *"Model too large for size-limited license"*. Two ways past it:



**In class — switch to HiGHS.** Open source, no licence, no size limit. Uncomment the `SOLVER` line below. There is no accuracy cost: HiGHS and Gurobi agree to sixteen significant figures on every model here. The speed cost is real only when the problem is large — measured on a 1-node model, HiGHS is *marginally faster* at 24 hours, identical at one week, and about **12x slower** on a full 8,760-hour year (22 s against 1.8 s). On a mixed-integer unit-commitment problem with 2,880 binary variables the gap was only **1.8x**.



**For homework — get an academic licence.** A free Web License Service (WLS) key from gurobi.com works in Colab with **no licence file**: paste the three values into `WLS` below. Build the environment **once** and reuse it — constructing a new one for every solve re-authenticates each time and will exhaust a WLS session partway through a scenario sweep.



In [3]:
SOLVER = 'gurobi'

# SOLVER = 'highs'     # <-- uncomment: open source, no licence, no size cap



# For homework: paste your academic Web License Service key here.

# Leave it empty and Gurobi falls back to its restricted licence.

WLS = {}   # {'WLSACCESSID': '...', 'WLSSECRET': '...', 'LICENSEID': 000000}



ENV = None

if SOLVER == 'gurobi' and WLS:

    import gurobipy as gp

    ENV = gp.Env(params=WLS)     # ONE environment, reused by every solve



print(f'solver: {SOLVER}'

      + ('  (academic WLS licence)' if ENV else '  (default licence)'))



solver: gurobi  (default licence)


---
## Part 1 — The interchange

The first real task in any model comparison is getting the *same system* into both tools. That means writing your inputs down in a form that belongs to neither.

Five tables is enough for a capacity-expansion problem, and they map directly onto the five-part formulation this course requires:

| Table | Formulation part |
|---|---|
| `demand` | parameters — 24 hours × 5 nodes of load |
| `tech` | parameters — capital, lifetime, marginal cost per technology |
| `limit` | constraints — buildable MW per node and technology |
| `avail` | parameters — hourly capacity factor per technology |
| `arcs` | parameters — corridors, existing capacity, expansion cost |

**Neither model reads anything else.** If a number is not in these tables, it is a convention baked into one of the two tools, and finding it is the point of Part 3.


In [4]:
def export_tables(wacc=0.07, gas_mc=GAS_MC):
    """The tool-neutral bundle.  Write it to CSV and both models read it."""
    shape, solar_cf, wind_cf = profiles()

    demand = pd.DataFrame({h: shape * PEAK_MW * SHARE[h] for h in HUBS},
                          index=range(HOURS))

    tech = pd.DataFrame([
        dict(tech='wind',  capex=1_300_000, life=30, mc=0.0),
        dict(tech='solar', capex=  900_000, life=30, mc=0.0),
        dict(tech='gas',   capex=1_050_000, life=30, mc=gas_mc),
    ]).set_index('tech')
    tech['daily_capital'] = [daily_capital(r.capex, r.life, wacc)
                             for _, r in tech.iterrows()]

    limit = pd.DataFrame(0.0, index=HUBS, columns=list(tech.index))
    for hub, mw in WIND_MAX.items():
        limit.loc[hub, 'wind'] = mw
    for hub, mw in SOLAR_MAX.items():
        limit.loc[hub, 'solar'] = mw
    for hub, mw in GAS_MAX.items():
        limit.loc[hub, 'gas'] = mw

    avail = pd.DataFrame({'wind': wind_cf, 'solar': solar_cf,
                          'gas': np.ones(HOURS)}, index=range(HOURS))

    arcs = pd.DataFrame(ARCS, columns=['a', 'b', 'km'])
    arcs['existing_mw'] = EXISTING_ARC_MW
    arcs['max_mw'] = 12_000.0
    arcs['daily_capital'] = [daily_capital(LINE_CAPEX_PER_MW_KM * k,
                                           LINE_LIFE, wacc)
                             for k in arcs.km]
    return dict(demand=demand, tech=tech, limit=limit, avail=avail,
                arcs=arcs)


tables = export_tables()
print('tech:'); print(tables['tech'].round(2).to_string())
print('\nbuildable MW by node and technology:')
print(tables['limit'].astype(int).to_string())
print(f"\npeak demand {tables['demand'].sum(axis=1).max():,.0f} MW")


tech:
         capex  life     mc  daily_capital
tech                                      
wind   1300000    30   0.00         287.02
solar   900000    30   0.00         198.71
gas    1050000    30  22.14         231.82

buildable MW by node and technology:
                    wind  solar    gas
West Texas         22000  18000      0
Dallas-Fort Worth   3000   4000  12000
Houston                0   3000  30000
Austin                 0   2000      0
San Antonio         4000   4000      0

peak demand 45,000 MW


---
## Part 2 — Model A: PyPSA

Two variants, because one of them is needed later to isolate structure:

- **`dc`** — corridors are `Line` components, so PyPSA imposes Kirchhoff's voltage law: flow is set by impedance and cannot be routed.
- **`transport`** — corridors are `Link` components with `p_min_pu = -1`, so flow is limited only by capacity and *can* be routed. This is the transportation formulation of Chapter 17, and it is what most capacity-expansion tools outside power-systems software actually use.

Everything else is identical between the two.


In [5]:
def pypsa_model(t, network='dc'):
    n = pypsa.Network()
    n.set_snapshots(pd.RangeIndex(HOURS, name='hour'))
    n.add('Carrier', 'AC')
    for tech in t['tech'].index:
        n.add('Carrier', tech)
    for hub in HUBS:
        n.add('Bus', hub, carrier='AC')

    for _, r in t['arcs'].iterrows():
        name = f'{r.a[:3].upper()}_{r.b[:3].upper()}'
        if network == 'dc':
            n.add('Line', name, bus0=r.a, bus1=r.b, length=r.km,
                  x=0.0001 * r.km,
                  s_nom=r.existing_mw, s_nom_min=r.existing_mw,
                  s_nom_max=r.max_mw, s_nom_extendable=True,
                  capital_cost=r.daily_capital)
        else:
            n.add('Link', name, bus0=r.a, bus1=r.b, length=r.km,
                  p_min_pu=-1.0,          # bidirectional
                  p_nom=r.existing_mw, p_nom_min=r.existing_mw,
                  p_nom_max=r.max_mw, p_nom_extendable=True,
                  capital_cost=r.daily_capital)

    for hub in HUBS:
        n.add('Load', f'{hub}_demand', bus=hub,
              p_set=t['demand'][hub].values)
        for tech, row in t['tech'].iterrows():
            cap = t['limit'].loc[hub, tech]
            if cap <= 0:
                continue
            n.add('Generator', f'{hub}_{tech}', bus=hub, carrier=tech,
                  p_nom_extendable=True, p_nom_max=cap,
                  p_max_pu=pd.Series(t['avail'][tech].values,
                                     index=n.snapshots),
                  capital_cost=row.daily_capital, marginal_cost=row.mc)
    return n


def solve_pypsa(n):
    status, condition = n.optimize(solver_name=SOLVER, env=ENV,
                                   log_to_console=False)
    assert condition == 'optimal', condition
    return n.objective


pypsa_dc = pypsa_model(tables, 'dc')
obj_dc = solve_pypsa(pypsa_dc)
pypsa_tr = pypsa_model(tables, 'transport')
obj_tr = solve_pypsa(pypsa_tr)
print(f'PyPSA, DC power flow   ${obj_dc:,.2f} / day')
print(f'PyPSA, transport       ${obj_tr:,.2f} / day')


C:\Users\jonesec\dev\venvs\esm\Lib\site-packages\pypsa\network\io.py:2082: FutureWarning: pandas infers the `str` dtype for string data since its version 3.0. PyPSA still converts it back to numpy object dtype on import, but will keep it from PyPSA 2.0 on. Set `pypsa.options.api.legacy_string_dtype` explicitly to suppress this warning.
  new_static = _coerce_string_dtypes(new_static)


C:\Users\jonesec\AppData\Local\Temp\ipykernel_26936\2081971592.py:41: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  status, condition = n.optimize(solver_name=SOLVER, env=ENV,
Index(['WES_DAL', 'WES_AUS', 'DAL_HOU', 'DAL_AUS', 'HOU_AUS', 'AUS_SAN',
       'HOU_SAN'],
      dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.07s


...

INFO:gurobipy:Solved in 318 iterations and 0.01 seconds (0.00 work units)


INFO:gurobipy:Optimal objective  2.325071399e+07


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 426 primals, 970 duals
Objective: 2.33e+07
Solver: gurobi
Runtime: 0.01s
Dual bound: 2.33e+07
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper were not assigned to the network.


PyPSA, DC power flow   $23,257,917.95 / day
PyPSA, transport       $23,250,713.99 / day


---
## Part 3 — Model B: the same problem, written from the algebra in PuLP

This is the assignment. Do not translate PyPSA's code — write the formulation out in the five parts the course requires and then type *that* in. If you paraphrase PyPSA you will reproduce PyPSA's conventions without knowing which ones you inherited, and the comparison will be worthless.

**Sets** — nodes *i*, technologies *g*, hours *h*, arcs *a = (i, j)*.

**Parameters** — `daily_capital[g]`, `mc[g]`, `limit[i,g]`, `avail[g,h]`, `demand[i,h]`, `arc_capital[a]`, `existing[a]`, `arc_max[a]`.

**Decision variables** — `cap[i,g] ≥ 0` built capacity; `disp[i,g,h] ≥ 0` dispatch; `acap[a] ≥ 0` corridor capacity; `flow[a,h]` free (signed).

**Objective** — minimise

&nbsp;&nbsp;&nbsp;&nbsp;Σ<sub>i,g</sub> daily_capital[g]·cap[i,g] &nbsp;+&nbsp; Σ<sub>a</sub> arc_capital[a]·acap[a] &nbsp;+&nbsp; Σ<sub>i,g,h</sub> mc[g]·disp[i,g,h]

**Constraints** —

- build limit: cap[i,g] ≤ limit[i,g]
- availability: disp[i,g,h] ≤ avail[g,h]·cap[i,g]
- corridor bounds: existing[a] ≤ acap[a] ≤ arc_max[a], and −acap[a] ≤ flow[a,h] ≤ acap[a]
- **nodal balance**: Σ<sub>g</sub> disp[i,g,h] + inflow − outflow = demand[i,h]

Note what is *not* there: no Kirchhoff voltage law. This is a transport model, so it should be compared against PyPSA's `transport` variant, not its `dc` one. Comparing it to the DC run would confound structure with everything else.


In [6]:
def pulp_model(t, charge_existing=True):
    """Capacity expansion as a transport LP, built from the formulation.

    charge_existing : whether corridor capital is charged on the whole of
        acap or only on what is built above the existing capacity.  This is
        the convention Part 4 is about; leave it True for the first run.
    """
    T = list(range(HOURS))
    G = list(t['tech'].index)
    arcs = t['arcs'].set_index(['a', 'b'])
    A_ = list(arcs.index)

    m = pulp.LpProblem('capacity_expansion', pulp.LpMinimize)
    cap = pulp.LpVariable.dicts('cap', (HUBS, G), lowBound=0)
    disp = pulp.LpVariable.dicts('disp', (HUBS, G, T), lowBound=0)
    acap = pulp.LpVariable.dicts('acap', A_, lowBound=0)
    flow = pulp.LpVariable.dicts('flow', (A_, T))     # signed, a -> b

    arc_cost = pulp.lpSum(
        arcs.daily_capital[a] * (acap[a] if charge_existing
                                 else acap[a] - arcs.existing_mw[a])
        for a in A_)
    m += (pulp.lpSum(t['tech'].daily_capital[g] * cap[i][g]
                     for i in HUBS for g in G)
          + arc_cost
          + pulp.lpSum(t['tech'].mc[g] * disp[i][g][h]
                       for i in HUBS for g in G for h in T))

    for i in HUBS:
        for g in G:
            m += cap[i][g] <= t['limit'].loc[i, g]
            for h in T:
                m += disp[i][g][h] <= t['avail'][g][h] * cap[i][g]
    for a in A_:
        m += acap[a] >= arcs.existing_mw[a]
        m += acap[a] <= arcs.max_mw[a]
        for h in T:
            m += flow[a][h] <= acap[a]
            m += flow[a][h] >= -acap[a]
    for i in HUBS:
        for h in T:
            inflow = pulp.lpSum(flow[a][h] for a in A_ if a[1] == i)
            outflow = pulp.lpSum(flow[a][h] for a in A_ if a[0] == i)
            m += (pulp.lpSum(disp[i][g][h] for g in G)
                  + inflow - outflow == t['demand'][i][h])
    return m, cap, acap


def solve_pulp(m):
    m.solve(pulp.HiGHS(msg=False))
    assert pulp.LpStatus[m.status] == 'Optimal', pulp.LpStatus[m.status]
    return pulp.value(m.objective)


m, cap, acap = pulp_model(tables)
obj_pulp = solve_pulp(m)
print(f'PuLP, transport        ${obj_pulp:,.2f} / day')
print(f'PyPSA, transport       ${obj_tr:,.2f} / day')
print(f'difference             ${obj_pulp - obj_tr:,.2f}  ({100*(obj_pulp-obj_tr)/obj_tr:+.2f}%)')


PuLP, transport        $24,930,096.69 / day
PyPSA, transport       $23,250,713.99 / day
difference             $1,679,382.69  (+7.22%)


---
## Part 4 — Account for the difference

The two models disagree. Before reaching for physics, check whether they built different things — because if the *builds* are identical and only the *cost* differs, no optimisation decision changed and the difference is pure accounting.


In [7]:
pypsa_build = (pypsa_tr.generators.p_nom_opt
               .groupby(pypsa_tr.generators.carrier).sum())
pulp_build = pd.Series({g: sum(cap[i][g].value() for i in HUBS)
                        for g in tables['tech'].index})
compare = pd.DataFrame({'PyPSA': pypsa_build, 'PuLP': pulp_build})
compare['difference'] = compare.PyPSA - compare.PuLP
print('generation capacity built (MW):')
print(compare.round(1).to_string())

arc_cmp = pd.DataFrame({
    'PyPSA': pypsa_tr.links.p_nom_opt.values,
    'PuLP': [acap[a].value() for a in tables['arcs']
             .set_index(['a', 'b']).index],
}, index=pypsa_tr.links.index)
arc_cmp['difference'] = arc_cmp.PyPSA - arc_cmp.PuLP
print('\ncorridor capacity (MW):')
print(arc_cmp.round(1).to_string())


generation capacity built (MW):
         PyPSA     PuLP  difference
gas    27000.0  27000.0         0.0
solar  19466.9  19466.9         0.0
wind       0.0      0.0         0.0

corridor capacity (MW):
          PyPSA    PuLP  difference
name                               
WES_DAL  3000.0  3000.0         0.0
WES_AUS  3000.0  3000.0         0.0
DAL_HOU  3000.0  3000.0         0.0
DAL_AUS  3000.0  3000.0         0.0
HOU_AUS  3000.0  3000.0         0.0
AUS_SAN  3000.0  3000.0         0.0
HOU_SAN  3000.0  3000.0         0.0


### The builds are identical. So where does the money go?

Nothing the solver decided is different, so the gap has to be a term one objective contains and the other does not. There is exactly one candidate: the **3,000 MW of corridor capacity that already exists**.

PyPSA charges `capital_cost` only on capacity built *above* an extendable asset's existing `s_nom`/`p_nom` — existing capacity is treated as sunk. The PuLP model as written charges the whole of `acap`, existing capacity included.

Test it: the gap should equal the daily capital charge on 3,000 MW across the seven corridors, exactly.


In [8]:
sunk = (EXISTING_ARC_MW * tables['arcs'].daily_capital).sum()
gap = obj_pulp - obj_tr
print(f'observed gap                            ${gap:>14,.2f}')
print(f'daily capital on 3,000 MW x 7 corridors ${sunk:>14,.2f}')
print(f'residual                                ${gap - sunk:>14,.2f}')
assert abs(gap - sunk) < 1.0, 'the gap is NOT just the sunk-cost convention'

# Align the convention and re-solve.
m2, cap2, acap2 = pulp_model(tables, charge_existing=False)
obj_pulp2 = solve_pulp(m2)
print(f'\nPuLP with PyPSA\'s sunk-cost convention  ${obj_pulp2:>14,.2f}')
print(f'PyPSA, transport                        ${obj_tr:>14,.2f}')
print(f'residual difference                     ${obj_pulp2 - obj_tr:>14,.2f}')


observed gap                            $  1,679,382.69
daily capital on 3,000 MW x 7 corridors $  1,679,382.69
residual                                $         -0.00



PuLP with PyPSA's sunk-cost convention  $ 23,250,713.99
PyPSA, transport                        $ 23,250,713.99
residual difference                     $         -0.00


> **Exercise 4.1.** Neither convention is wrong. Say which one you would use for **(a)** a report recommending whether to build a new corridor and **(b)** a report comparing the total cost of two whole system designs, and why the answer differs between the two.

> **Exercise 4.2.** The gap was 7% of system cost, and it came from a convention neither model documents in its output. Describe the check you would run *first*, on any future model comparison, to catch this class of difference before you start looking for real ones. (You have just seen it: compare the decision variables before comparing the objective.)


---
## Part 5 — Isolate structure, then isolate data

With the conventions aligned, the two arms of the decomposition can be measured separately.

### Structure, with the data held fixed

`PyPSA-DC` against `PyPSA-transport` is the cleanest structural experiment available: same tool, same tables, same solver, one difference — whether Kirchhoff's voltage law is imposed. Any gap is the cost of not being able to route power.


In [9]:
delta_structure = obj_dc - obj_tr
print(f'PyPSA, DC power flow   ${obj_dc:>14,.2f}')
print(f'PyPSA, transport       ${obj_tr:>14,.2f}')
print(f'cost of KVL            ${delta_structure:>14,.2f}   ({100*delta_structure/obj_tr:+.3f}%)')

dc_arcs = pypsa_dc.lines.s_nom_opt
tr_arcs = pypsa_tr.links.p_nom_opt
extra = (dc_arcs - tr_arcs).round(1)
print('\nextra corridor capacity the DC model has to build (MW):')
print(extra[extra.abs() > 0.5].to_string() or '  none')


PyPSA, DC power flow   $ 23,257,917.95
PyPSA, transport       $ 23,250,713.99
cost of KVL            $      7,203.95   (+0.031%)

extra corridor capacity the DC model has to build (MW):
name
HOU_AUS    118.1
HOU_SAN      4.8


### Data, with the structure held fixed

Now vary the tables and re-solve **both** models. If the interchange is faithful, the two move together under every scenario and the residual stays at zero.


In [10]:
scenarios = [
    ('base (WACC 7%, gas $22.14/MWh)', {}),
    ('WACC 10%', dict(wacc=0.10)),
    ('gas $45/MWh', dict(gas_mc=45.0)),
    ('WACC 10% and gas $45/MWh', dict(wacc=0.10, gas_mc=45.0)),
]
rows = []
for label, kw in scenarios:
    t = export_tables(**kw)
    n = pypsa_model(t, 'transport')
    a = solve_pypsa(n)
    mm, cc, _ = pulp_model(t, charge_existing=False)
    b = solve_pulp(mm)
    built = n.generators.p_nom_opt.groupby(n.generators.carrier).sum()
    rows.append({
        'scenario': label,
        'PyPSA $': round(a, 2), 'PuLP $': round(b, 2),
        'residual $': round(b - a, 2),
        'wind MW': round(built.get('wind', 0.0)),
        'solar MW': round(built.get('solar', 0.0)),
        'gas MW': round(built.get('gas', 0.0)),
    })
print(pd.DataFrame(rows).to_string(index=False))


C:\Users\jonesec\AppData\Local\Temp\ipykernel_26936\2081971592.py:41: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  status, condition = n.optimize(solver_name=SOLVER, env=ENV,


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.05s


Read LP format model from file C:\Users\jonesec\AppData\Local\Temp\linopy-problem-aswdpdp0.lp


...

INFO:gurobipy:Solved in 313 iterations and 0.01 seconds (0.00 work units)


INFO:gurobipy:Optimal objective  3.778380497e+07


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 426 primals, 970 duals
Objective: 3.78e+07
Solver: gurobi
Runtime: 0.01s
Dual bound: 3.78e+07
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper were not assigned to the network.


                      scenario     PyPSA $      PuLP $  residual $  wind MW  solar MW  gas MW
base (WACC 7%, gas $22.14/MWh) 23250713.99 23250713.99        -0.0        0     19467   27000
                      WACC 10% 26453163.38 26453163.38        -0.0        0     19260   27050
                   gas $45/MWh 31602985.93 31602985.93         0.0    29000     31000   18883
      WACC 10% and gas $45/MWh 37783804.97 37783804.97         0.0    29000     27590   18883


### Read the table before moving on

Three things should be visible in it.

1. **The residual is zero in every scenario.** The interchange is faithful: whatever else differs between these two models, it is not the data.
2. **The data assumptions move the answer a great deal.** Gas at $45/MWh instead of $22.14 brings tens of gigawatts of wind into a build that had none. That is not a modelling difference at all — it is one number in one table.
3. **Set that against the structural effect from the previous cell.** Kirchhoff's voltage law was worth a fraction of a percent. The accounting convention in Part 4 was worth seven percent. The fuel price is worth more than either.

That ordering is the result of this assignment, and it is not what most people expect. **The physics you argued about mattered least.**


---
## Part 6 — The full-weight route: PowerGenome → GenX

If you want the version a research group would run, this is it. It does not fit in a notebook cell, and it is a legitimate substitute for Parts 2–5 if you would rather do the decomposition across two genuinely separate tools.

**GenX** (`github.com/GenXProject/GenX`) is a Julia capacity-expansion model from MIT. It reads a directory of CSVs — `Generators_data.csv`, `Load_data.csv`, `Generators_variability.csv`, `Network.csv`, `Fuels_data.csv` — which is the same interchange idea as Part 1, with a fixed schema. Install Julia, then `] add GenX`, and run `run_genx_case!`.

**PowerGenome** (`github.com/PowerGenome/PowerGenome`) is the data pipeline that produces those CSVs from public sources — EIA 860/923, NREL ATB, EPA. It is a Python package with a YAML settings file, and it is where most of the work is: deciding how to cluster generators, which vintages to retain, what to do with must-run units.

**OSeMOSYS** (`osemosys.readthedocs.io`) has PuLP and Pyomo implementations that read a single tabular data file. Use the PuLP or Pyomo versions — the GAMS version needs a commercial licence and is excluded from this course.

### If you take this route

The deliverable does not change: **decompose the difference**. But the decomposition is harder, because the two tools do not share a formulation, so you must account for at least these before you can claim a structural difference:

- time resolution and representative-period weighting
- whether either tool applies a reserve margin or planning-reserve constraint by default
- unit commitment — GenX has commitment options PyPSA's LP relaxation does not
- the same sunk-versus-total capital convention you met in Part 4
- retirement and vintage handling

Each of those is a *structural* difference. None is a difference in the system. Say which ones you controlled for and which you could not.


---
## What to hand in

1. **The interchange.** The five tables, and one sentence on what a table does *not* carry — a convention you found in one tool that had to be set by hand in the other.
2. **The second model.** Your own formulation, in the five parts, and the code that implements it. If you paraphrased the first model's code instead of writing the formulation, say so; it changes what the comparison means.
3. **The reconciliation.** Every dollar of difference between the two answers, attributed. Do not report "the models broadly agree" — report the residual and what it is.
4. **Where the optimal build differs**, and the cause split into a **data-assumption** difference and a **model-structure** difference, with the number for each. Parts 4 and 5 give you the method; the numbers are yours.
5. **One thing your second tool makes easier to express than PyPSA, and one thing it makes harder.** Be concrete. "PuLP made the objective explicit, so I could see the sunk-cost convention that PyPSA applies silently; PyPSA made the DC power flow one keyword, and reproducing Kirchhoff's voltage law by hand in PuLP would have meant building the cycle basis myself" is the level of specificity expected.

### The trap to avoid

A comparison that ends in "the two models gave different answers, which shows the importance of model choice" has not done the assignment. Any two models give different answers. **The assignment is the accounting.**


---

*Before class: these two pipelines disagree. Which would you trust to size a single site's connection, and what is the other one actually answering?*

### Sources
- **GenX** — github.com/GenXProject/GenX. Jenkins & Sepulveda, MIT Energy Initiative
- **PowerGenome** — github.com/PowerGenome/PowerGenome
- **OSeMOSYS** — osemosys.readthedocs.io. Use the PuLP or Pyomo implementation; the GAMS version requires a commercial licence
- **PuLP** — coin-or.github.io/pulp
- **PyPSA** — pypsa.readthedocs.io
- Cost basis and hub demand shares as in the SB6 notebook: NREL ATB 2024, EIA 2023, and ERCOT weather-zone shares computed from TX-123BT (Jin Lu et al., DOI 10.6084/m9.figshare.22144616, CC BY 4.0)
